# **Mental Health Support Chatbot using coustom PDF data**

**Install initial Dependecies**

In [ ]:
!pip install langchain_groq langchain_core langchain_community

###### **Test The Api**

In [ ]:
from langchain_groq import ChatGroq
llm = ChatGroq(
    temperature = 0,
    groq_api_key = "Enter_Your_GroqAPIKey_Here",
    model_name = "llama-3.3-70b-versatile"
)
result = llm.invoke("Who is lord Ram?")
print(result.content)

In [ ]:
!pip install pypdf

In [ ]:
!pip install chromadb

In [ ]:
!pip install sentence_transformers

In [ ]:
!pip install -U langchain-huggingface

### **Add the PDF to chat and act like Doctor**

In [ ]:
import os
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq

def initialize_llm():

    llm = ChatGroq(
        temperature=0,
        groq_api_key="Enter_Your_GroqAPIKey_Here",
        model_name="llama-3.3-70b-versatile"
    )
    return llm

def create_vector_db():
    # Ensure the data directory exists
    data_path = "./data/"
    if not os.path.exists(data_path):
        print(f"Error: Path {data_path} does not exist. Please create it and add PDFs.")
        return None

    loader = DirectoryLoader(data_path, glob='*.pdf', loader_cls=PyPDFLoader)
    documents = loader.load()

    # Splitting text into chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    texts = text_splitter.split_documents(documents)

    # Updated embedding class
    embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

    # Create and persist the vector store
    vector_db = Chroma.from_documents(
        documents=texts,
        embedding=embeddings,
        persist_directory='./chroma_db'
    )

    print("ChromaDB created and data saved locally.")
    return vector_db

def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

def setup_rag_chain(vector_db, llm):
    retriever = vector_db.as_retriever(search_kwargs={"k": 3})

    prompt_template = """You are a compassionate mental health chatbot.
Use the following context to help answer the user's question thoughtfully.
If you don't know the answer based on the context, say that you don't know, but remain supportive.

Context: {context}

User: {question}
Chatbot:"""

    prompt = PromptTemplate.from_template(prompt_template)

    # Building the chain using LCEL (Modern replacement for RetrievalQA)
    rag_chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    return rag_chain

def main():
    print("Initializing Chatbot...")
    llm = initialize_llm()
    db_path = "./chroma_db"

    # Initialize embeddings
    embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')

    if not os.path.exists(db_path):
        print("Vector database not found. Creating new database from PDFs...")
        vector_db = create_vector_db()
        if vector_db is None:
            return
    else:
        print("Loading existing Vector database...")
        vector_db = Chroma(persist_directory=db_path, embedding_function=embeddings)

    # Setup the modern RAG chain
    rag_chain = setup_rag_chain(vector_db, llm)

    print("\n--- Mental Health Support Chatbot Active (type 'exit' to quit) ---")

    while True:
        query = input("\nHuman: ")
        if query.lower() in ["exit", "quit", "bye"]:
            print("Chatbot: Take care of yourself. Goodbye!")
            break

        try:
            # Execute the chain
            response = rag_chain.invoke(query)
            print(f"Chatbot: {response}")
        except Exception as e:
            print(f"Chatbot: I encountered an error: {e}")

if __name__ == "__main__":
    main()

#### **Run this Dependecies to install at once and latest version**

In [ ]:
!pip install -U --force-reinstall langchain langchain-community langchain-core langchain-text-splitters langchain-groq sentence-transformers chromadb gradio pypdf

In [ ]:
!python.exe -m pip install --upgrade pip

In [ ]:
!pip install gradio

In [ ]:
!pip install "langchain[all]" langchain-groq chromadb

### **Final ChatBot add Gui And launch it**

In [ ]:
import os
import gradio as gr


from langchain_community.embeddings import HuggingFaceBgeEmbeddings
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_community.vectorstores import Chroma

from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_groq import ChatGroq

def initialize_llm():

    # Consider replacing this with os.environ.get("GROQ_API_KEY") in the future.
    llm = ChatGroq(
        temperature=0,
        groq_api_key="Enter_Your_GroqAPIKey_Here",
        model_name="llama-3.3-70b-versatile"
    )
    return llm

def create_vector_db():
    loader = DirectoryLoader("/data/", glob='*.pdf', loader_cls=PyPDFLoader)
    documents = loader.load()
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    texts = text_splitter.split_documents(documents)

    embeddings = HuggingFaceBgeEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    vector_db = Chroma.from_documents(texts, embeddings, persist_directory='./chroma_db')

    print("ChromaDB created and data saved")
    return vector_db

def setup_qa_chain(vector_db, llm):
    retriever = vector_db.as_retriever()
    prompt_templates = """You are a compassionate mental health chatbot. Respond thoughtfully to the following question:
    {context}
    User: {question}
    Chatbot: """
    PROMPT = PromptTemplate(template=prompt_templates, input_variables=['context', 'question'])

    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        chain_type_kwargs={"prompt": PROMPT}
    )
    return qa_chain


print("Initializing Chatbot.........")
llm = initialize_llm()

db_path = "./chroma_db"

if not os.path.exists(db_path):
    vector_db = create_vector_db()
else:
    embeddings = HuggingFaceBgeEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    vector_db = Chroma(persist_directory=db_path, embedding_function=embeddings)

qa_chain = setup_qa_chain(vector_db, llm)

def chatbot_response(message, history):
    if not message.strip():
        return "Please provide a valid input"

    # Gradio's ChatInterface expects a simple string return
    response = qa_chain.run(message)
    return response

with gr.Blocks(theme='Respair/Shiki@1.2.1') as app:
    gr.Markdown("# 🧠 Mental Health Chatbot 🤖")
    gr.Markdown("A compassionate chatbot designed to assist with mental well-being. Please note: For serious concerns, contact a professional.")

    chatbot = gr.ChatInterface(fn=chatbot_response, title="Mental Health Chatbot")

    gr.Markdown("This chatbot provides general support. For urgent issues, seek help from licensed professionals.")

app.launch(debug=True)

**After launching the cell a link will be generated for gui which should look like this "http://127.0.0.1:7860/" this link
will give you access of your gui
and Replace with your groq api key**